# StashFace — الخطوة 1: بناء قاعدة الوجوه (Build) على Kaggle (2x T4 GPU)

**قبل ما تشغل أي خلية:** من إعدادات الـnotebook (يمين الشاشة) اختار Accelerator = **GPU T4 x2**.

شغّل الخلايا بالترتيب من فوق لتحت. لما تخلص، روح لـnotebook التاني `stashface_query.ipynb` عشان تعمل المقارنة (مش محتاج GPU خالص).

In [ ]:
# ============================================================
# خلية الإعداد — عدّل القيم دي بس، والباقي متلمسوش
# ============================================================

# رابط الـGitHub repo بتاع المشروع
GITHUB_REPO_URL = "https://github.com/kareemkamal10/stashface_pipeline"

# الـtoken بتاع حسابك على Hugging Face (لازم يكون عنده صلاحية Write)
HF_TOKEN = " "

# الـdataset اللي فيه ملفات الأداء، وهيتحفظ فيه كل تقارير الفحص في الآخر
HF_DATASET_ID = "abdelwahabnabil500/datafile"

# اسم الملفين جوه الـdataset (نفس الاسمين المحليين بالظبط)
HF_INPUT_WITH_TPDB = "performers_with_tpdb.json"
HF_INPUT_WITHOUT_TPDB = "performers_without_tpdb.json"

In [ ]:
# ============================================================
# تحميل الكود وتثبيت المكتبات — مفيش حاجة تتعدل هنا
# ============================================================
import os

# منع hf CLI من سؤال "تحدّث دلوقتي؟" اللي بيعلّق جوه notebook (مفيش حد يرد عليه)
os.environ["HF_HUB_DISABLE_UPDATE_CHECK"] = "1"

# 1) تحميل الكود من الـGitHub repo
!git clone {GITHUB_REPO_URL} /kaggle/temp/stashface_pipeline
%cd /kaggle/temp/stashface_pipeline

# 2) تثبيت المكتبات المطلوبة
!pip install -q -r requirements.txt
!pip install -q -U "huggingface_hub[cli]>=1.13.0"

# 3) استبدال onnxruntime بنسخة الـGPU — عشان يستخدم الـT4 فعليًا مش الـCPU
!pip uninstall -y -q onnxruntime
!pip install -q "onnxruntime-gpu==1.26.0"  # pinned: onnxruntime-gpu>=1.27 defaults to CUDA 13, Kaggle T4 images still run CUDA 12.x

# 4) تسجيل الدخول لـHugging Face بالـtoken بتاعك
from huggingface_hub import login
login(token=HF_TOKEN)
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
# ============================================================
# تحميل بيانات المشروع (الموديل) + ملفي الأداء
# ============================================================

# 5) تحميل بيانات المشروع (adaface model) من الـbucket
!python setup.py --skip-install

# 6) تحميل الملفين من الـdataset بتاعك
from huggingface_hub import hf_hub_download
import shutil

for remote_name, local_name in [
    (HF_INPUT_WITH_TPDB, "performers_with_tpdb.json"),
    (HF_INPUT_WITHOUT_TPDB, "performers_without_tpdb.json"),
]:
    local_path = hf_hub_download(
        repo_id=HF_DATASET_ID,
        repo_type="dataset",
        filename=remote_name,
        token=HF_TOKEN,
    )
    shutil.copy(local_path, local_name)
    print("تم تحميل:", local_name)

## بناء قاعدة الوجوه

دي هتاخد وقت طويل (ساعات، حسب حجم البيانات). لو الجلسة اتقفلت أو حصل أي كراش - أو حتى لو الجلسة كلها راحت وفتحت واحدة جديدة - رجّع شغّل نفس الخلايا تاني، هيكمل من آخر batch اترفع على HF بدل ما يبدأ من الصفر.

بيبني الـDB من الملفين مع بعض (تحميل + كشف وش لكل عنصر)، وبيرفع نسخة محدثة منها على HF بعد كل batch أول بأول.

In [ ]:
!python dedupe_pipeline.py --mode build --device-id 0 --hf-dataset-id {HF_DATASET_ID} --hf-token {HF_TOKEN}

## خلصت مرحلة البناء

القاعدة (`face_db.npz`) وتقارير الفشل اترفعوا على `reports/` جوا الـdataset. الخلية اللي جاية بس بتوريك ملخص محلي سريع.

**الخطوة التالية**: افتح `stashface_query.ipynb` (notebook تاني، مش محتاج GPU خالص) عشان تعمل المقارنة الفعلية.

In [ ]:
import json, os

for fname in [
    "dedupe_reports/face_db.npz",
    "dedupe_reports/download_failed.json",
    "dedupe_reports/no_face_detected.json",
]:
    if not os.path.exists(fname):
        print(f"{fname}: مش موجود")
        continue
    if fname.endswith(".json"):
        n = len(json.load(open(fname, encoding="utf-8")))
        print(f"{fname}: {n} سجل")
    else:
        size_mb = os.path.getsize(fname) / (1024 * 1024)
        print(f"{fname}: {size_mb:.1f} MB")